# BrandSphere AI — Week 4: Slogan & NLP Engine

Covers:
1. Text preprocessing with NLTK
2. TF-IDF vectorization on slogan corpus
3. Template-based generation
4. Gemini API enhancement
5. Slogan quality analysis

In [ ]:
import pandas as pd
import numpy as np
import re, string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('✅ Libraries loaded')

## 1. Load and Clean Slogan Corpus

In [ ]:
# Load startup taglines as our slogan corpus
df = pd.read_csv('../datasets/raw/startups.csv')
df = df.dropna(subset=['tagline'])
df['tagline'] = df['tagline'].str.strip()
df = df[df['tagline'].str.len() > 4]
print(f'Corpus size: {len(df):,} taglines')
df[['name','tagline']].head(10)

## 2. NLTK Text Preprocessing (Week 4 Core Requirement)

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    """Full NLTK preprocessing pipeline."""
    tokens = word_tokenize(text.lower())
    clean = [t for t in tokens if t not in stop_words and t not in string.punctuation]
    stems = [stemmer.stem(t) for t in clean]
    return {
        'tokens': tokens,
        'clean_tokens': clean,
        'stems': stems,
        'word_count': len(clean),
        'unique_words': len(set(clean)),
        'lexical_density': round(len(set(clean)) / max(1, len(clean)), 3)
    }

# Demo
example = 'Just Do It'
result = preprocess(example)
print(f'Input:           "{example}"')
for k, v in result.items():
    print(f'{k:20s}: {v}')

In [ ]:
# Apply to full corpus
df['slogan_clean'] = df['tagline'].str.lower().str.replace(r'[^a-z0-9\s]','',regex=True).str.strip()
df['word_count'] = df['tagline'].apply(lambda x: len(word_tokenize(x)))
df['clean_tokens'] = df['tagline'].apply(lambda x: preprocess(x)['clean_tokens'])
df['stems'] = df['tagline'].apply(lambda x: preprocess(x)['stems'])
df['lexical_density'] = df['tagline'].apply(lambda x: preprocess(x)['lexical_density'])

print('Preprocessing complete')
print(df[['tagline','word_count','lexical_density']].describe())

## 3. TF-IDF Similarity Retrieval

In [ ]:
# Fit TF-IDF on cleaned corpus
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=8000, min_df=2)
tfidf_matrix = vectorizer.fit_transform(df['slogan_clean'].fillna(''))
print(f'TF-IDF matrix: {tfidf_matrix.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')

In [ ]:
def retrieve_similar(query, top_k=5):
    """Retrieve most similar slogans from corpus."""
    q_vec = vectorizer.transform([query.lower()])
    sims = cosine_similarity(q_vec, tfidf_matrix)[0]
    idxs = np.argsort(sims)[::-1][:top_k]
    results = []
    for i in idxs:
        if sims[i] > 0.01:
            results.append({'slogan': df.iloc[i]['tagline'], 'similarity': round(float(sims[i]), 4)})
    return results

# Test retrieval
for query in ['innovative tech platform', 'luxury fashion brand', 'healthy food delivery']:
    print(f'\nQuery: "{query}"')
    for r in retrieve_similar(query, 3):
        print(f'  {r["similarity"]:.3f} — {r["slogan"]}')

## 4. Template-Based Generation

In [ ]:
TEMPLATES = {
    'Bold':          ['{company}: Lead. Don\'t follow.', 'Built to dominate {industry}.', '{company} — Fearless by design.'],
    'Minimalist':    ['{company}. Simply brilliant.', 'Less noise. More {company}.', 'Precision. Purpose. {company}.'],
    'Luxury':        ['Crafted for those who know — {company}.', 'The art of {industry}. {company}.', 'Rare by nature. Refined by choice.'],
    'Professional':  ['{company}: Delivering results that matter.', 'Your trusted {industry} partner.', 'Performance-driven. {company}.'],
    'Inspirational': ['Dream bigger. Build bolder. {company}.', '{company} — Where possibilities begin.', 'Your journey. Our mission.'],
}

def generate_from_template(company, industry, tone, n=3):
    templates = TEMPLATES.get(tone, TEMPLATES['Professional'])
    import random
    selected = random.sample(templates, min(n, len(templates)))
    return [t.replace('{company}', company).replace('{industry}', industry.lower()) for t in selected]

# Test
for tone in ['Bold', 'Minimalist', 'Luxury']:
    print(f'\n{tone}:')
    for s in generate_from_template('NovaTech', 'Technology', tone):
        print(f'  "{s}"')

## 5. Quality Analysis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Slogan Corpus Analysis', color='white')

df['word_count'].value_counts().sort_index().head(15).plot(kind='bar', ax=axes[0], color='#C9A84C')
axes[0].set_title('Word Count Distribution', color='white')

df['lexical_density'].hist(ax=axes[1], bins=20, color='#C9A84C', alpha=0.8)
axes[1].set_title('Lexical Density', color='white')

df['tagline'].str.len().hist(ax=axes[2], bins=30, color='#3ECFB2', alpha=0.8)
axes[2].set_title('Character Length', color='white')

for ax in axes: ax.tick_params(colors='gray')
plt.tight_layout()
plt.savefig('../assets/sample_exports/slogan_analysis.png', dpi=100)
plt.show()
print('✅ Analysis complete')

## Summary
- NLTK pipeline: tokenization → stopword removal → Porter stemming ✅
- TF-IDF retrieval on 36K+ startup taglines ✅
- Template-based generation for 5 tone variants ✅
- Gemini API integration in `src/slogan_engine.py` ✅